# HW12: Временные ряды — прогнозирование


In [1]:
import os, json, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.metrics import (mean_absolute_error,
                             mean_squared_error,
                             mean_absolute_percentage_error)
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
# Воспроизводимость
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

ARTIFACTS = Path('artifacts')
FIGURES   = ARTIFACTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

device: cpu


## 1. Загрузка и первичный анализ

In [2]:
df = pd.read_csv('S12-hw-dataset.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min()}  {df['date'].max()}")
print(f"Nulls:\n{df.isnull().sum()}")
print(f"\nDescribe:\n{df['target'].describe()}")

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(df['date'], df['target'], lw=0.7)
ax.set(title='Исходный временной ряд', xlabel='Дата', ylabel='Target')
ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES / 'raw_series.png', dpi=150)
plt.close(fig)
print("Saved raw_series.png")

Shape: (4320, 2)
Date range: 2025-01-01 00:00:00  2025-06-29 23:00:00
Nulls:
date      0
target    0
dtype: int64

Describe:
count    4320.000000
mean      135.605840
std        21.384633
min        69.100000
25%       120.537500
50%       135.835000
75%       150.625000
max       210.100000
Name: target, dtype: float64
Saved raw_series.png


**Анализ ряда:**
- тренд восходящий 
- в течение 6 месяцев наблюдается высокочастотная сезонность
- выраженных выбросов нет

## 2. Temporal Split

random split нельзя использовать так как
при случайном перемешивании будущие наблюдения попадают в train, а прошлые — в test, то есть модель будет завышать искуственно метрики 
Для временных рядов правиольное будет использовать хронологическое разбиение Temporal split

In [3]:
N = len(df)
n_val  = int(N * 0.15)
n_test = int(N * 0.15)
n_train = N - n_val - n_test

train_df = df.iloc[:n_train].copy()
val_df   = df.iloc[n_train:n_train+n_val].copy()
test_df  = df.iloc[n_train+n_val:].copy()

for name, part in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"{name:5s}: {len(part):5d} rows  "
          f"({part['date'].min().date()}  {part['date'].max().date()})")

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(train_df['date'], train_df['target'], label='Train',  lw=0.7)
ax.plot(val_df['date'],   val_df['target'],   label='Validation', lw=0.7)
ax.plot(test_df['date'],  test_df['target'],   label='Test',   lw=0.7)
ax.legend(); ax.set(title='Temporal Split', ylabel='Target')
ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES / 'series_split.png', dpi=150)
plt.close(fig)
print("Saved series_split.png")

Train:  3024 rows  (2025-01-01  2025-05-06)
Val  :   648 rows  (2025-05-07  2025-06-02)
Test :   648 rows  (2025-06-03  2025-06-29)
Saved series_split.png


## 3. Feature Engineering

Строим признаки **без утечки из будущего**:
- `lag_1`, `lag_7`, `lag_14`, `lag_24`, `lag_168` — авторегрессионные лаги
- `rolling_mean_24`, `rolling_std_24` — скользящие статистики (shift(1) перед rolling, чтобы не включать текущее значение)
- `hour`, `dayofweek` — календарные признаки

In [4]:
def make_features(data):
    d = data.copy()
    d['hour']      = d['date'].dt.hour
    d['dayofweek'] = d['date'].dt.dayofweek
    for lag in [1, 7, 14, 24, 168]:
        d[f'lag_{lag}'] = d['target'].shift(lag)

    d['rolling_mean_7'] = d['target'].shift(1).rolling(7).mean()
    d['rolling_std_7']  = d['target'].shift(1).rolling(7).std()

    d['rolling_mean_24'] = d['target'].shift(1).rolling(24).mean()
    d['rolling_std_24']  = d['target'].shift(1).rolling(24).std()
    
    return d

df_feat = make_features(df).dropna().reset_index(drop=True)

train_end = train_df['date'].iloc[-1]
val_end   = val_df['date'].iloc[-1]

train_f = df_feat[df_feat['date'] <= train_end]
val_f   = df_feat[(df_feat['date'] > train_end) & (df_feat['date'] <= val_end)]
test_f  = df_feat[df_feat['date'] > val_end]

FEATURE_COLS = [c for c in df_feat.columns if c not in ('date', 'target')]

X_train, y_train = train_f[FEATURE_COLS].values, train_f['target'].values
X_val,   y_val   = val_f[FEATURE_COLS].values,   val_f['target'].values
X_test,  y_test  = test_f[FEATURE_COLS].values,  test_f['target'].values

print(f"Features: {FEATURE_COLS}")
print(f"X_train {X_train.shape}, X_val {X_val.shape}, X_test {X_test.shape}")

Features: ['hour', 'dayofweek', 'lag_1', 'lag_7', 'lag_14', 'lag_24', 'lag_168', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_24', 'rolling_std_24']
X_train (2856, 11), X_val (648, 11), X_test (648, 11)


In [5]:
# Логирование экспериметов
experiments = []

def calc_metrics(y_true, y_pred):
    if y_true is None or y_pred is None:
        return {'mae': None, 'rmse': None, 'mape': None}
    return {
        'mae':  mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mape': mean_absolute_percentage_error(y_true, y_pred),
    }

def log_exp(exp_id, model_name, y_val_true, y_val_pred, y_test_true=None, y_test_pred=None,
            task='forecasting', dataset='S12-hw-dataset.csv', seed=SEED,
            split_summary='Train: 3024, Val: 648, Test: 648', window_size='',
            horizon=1, model_summary=None, features_summary='', scaler='',
            optimizer='', lr='', epochs_trained='', notes=''):
    
    m_val = calc_metrics(y_val_true, y_val_pred)
    m_test = calc_metrics(y_test_true, y_test_pred)

    experiments.append({
        'experiment_id': exp_id,
        'task': task,
        'dataset': dataset,
        'seed': seed,
        'split_summary': split_summary,
        'window_size': window_size,
        'horizon': horizon,
        'model_summary': model_summary or model_name,
        'features_summary': features_summary,
        'scaler': scaler,
        'optimizer': optimizer,
        'lr': lr,
        'epochs_trained': epochs_trained,
        'best_val_mae': round(m_val['mae'], 4) if m_val['mae'] is not None else '',
        'best_val_rmse': round(m_val['rmse'], 4) if m_val['rmse'] is not None else '',
        'best_val_mape': round(m_val['mape'], 6) if m_val['mape'] is not None else '',
        'test_mae': round(m_test['mae'], 4) if m_test['mae'] is not None else '',
        'test_rmse': round(m_test['rmse'], 4) if m_test['rmse'] is not None else '',
        'test_mape': round(m_test['mape'], 6) if m_test['mape'] is not None else '',
        'notes': notes,
    })
    print(f"[{exp_id}] {model_name:25s} | "
          f"Val MAE={m_val['mae']:.4f}  RMSE={m_val['rmse']:.4f}  MAPE={m_val['mape']:.4f}")


## 4. Baseline-эксперименты

### B1 — Naive last value

In [6]:
y_pred_b1 = val_f['lag_1'].values
y_pred_test_b1 = test_f['lag_1'].values
log_exp('B1', 'Naive (lag_1)', y_val, y_pred_b1, y_test, y_pred_test_b1,
        window_size=1, features_summary='lag_1', model_summary='Naive Last Value')


[B1] Naive (lag_1)             | Val MAE=6.4448  RMSE=8.2010  MAPE=0.0440


### B2 — Moving Average 24 h


In [7]:
y_pred_b2 = val_f['rolling_mean_24'].values
y_pred_test_b2 = test_f['rolling_mean_24'].values
log_exp('B2', 'Moving Average 24h', y_val, y_pred_b2, y_test, y_pred_test_b2,
        window_size=24, features_summary='rolling_mean_24', model_summary='Moving Average')


[B2] Moving Average 24h        | Val MAE=13.3980  RMSE=16.1699  MAPE=0.0920


### B3 — Ridge Regression
Линейная модель на lag / rolling / calendar признаках. Scaler обучается **только на train**!

In [8]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_sc, y_train)

y_pred_b3 = ridge.predict(X_val_sc)
y_pred_test_b3 = ridge.predict(X_test_sc)
log_exp('B3', 'Ridge', y_val, y_pred_b3, y_test, y_pred_test_b3,
        features_summary='lag (1,7,14,24,168), rolling (mean_7, std_7, mean_24, std_24), calendar (hour, dayofweek)',
        scaler='StandardScaler', model_summary='Ridge(alpha=1.0)')

[B3] Ridge                     | Val MAE=4.6948  RMSE=6.0877  MAPE=0.0315


## 5. Эксперимент R1 — GRU Forecast

Используем оконное (windowed) представление: вход — последние `WINDOW=72` часа (3 суток),
выход — следующее значение. Масштабирование **fit на train**, transform на val/test.

In [9]:
WINDOW = 72  
BATCH  = 256
LR     = 1e-3
EPOCHS = 30

tgt_scaler = StandardScaler()
train_scaled = tgt_scaler.fit_transform(train_df[['target']].values).flatten()
val_scaled   = tgt_scaler.transform(val_df[['target']].values).flatten()
test_scaled  = tgt_scaler.transform(test_df[['target']].values).flatten()

class WindowDataset(Dataset):
    def __init__(self, data, window):
        self.data = torch.FloatTensor(data)
        self.w = window
    def __len__(self):
        return len(self.data) - self.w
    def __getitem__(self, i):
        x = self.data[i:i+self.w].unsqueeze(-1)   
        y = self.data[i+self.w]
        return x, y

train_ds = WindowDataset(train_scaled, WINDOW)
val_full = np.concatenate([train_scaled[-WINDOW:], val_scaled])
val_ds   = WindowDataset(val_full, WINDOW)

test_full = np.concatenate([val_scaled[-WINDOW:], test_scaled])
test_ds   = WindowDataset(test_full, WINDOW)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader  = DataLoader(val_ds,   batch_size=BATCH, shuffle=False)
test_loader = DataLoader(test_ds,  batch_size=BATCH, shuffle=False)

print(f"Train samples: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Train samples: 2952, Val: 648, Test: 648


In [10]:
class GRUForecaster(nn.Module):
    def __init__(self, inp=1, hid=64, layers=2, drop=0.2):
        super().__init__()
        self.gru = nn.GRU(inp, hid, layers, batch_first=True,
                          dropout=drop if layers > 1 else 0)
        self.fc  = nn.Linear(hid, 1)
    def forward(self, x):
        out, _ = self.gru(x)      
        return self.fc(out[:, -1])   

set_seed()
model = GRUForecaster().to(device)
opt   = torch.optim.Adam(model.parameters(), lr=LR)
crit  = nn.MSELoss()

gru_config = dict(input_dim=1, hidden_dim=64, num_layers=2,
                  dropout=0.2, window=WINDOW, lr=LR, epochs=EPOCHS, seed=SEED)
with open(ARTIFACTS / 'best_gru_config.json', 'w') as f:
    json.dump(gru_config, f, indent=2)
print("Model:", model)

Model: GRUForecaster(
  (gru): GRU(1, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [11]:
train_losses, val_losses = [], []
best_val = float('inf')

for ep in range(1, EPOCHS+1):
    # Train
    model.train()
    s = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(model(xb).squeeze(), yb)
        loss.backward(); opt.step()
        s += loss.item() * len(xb)
    train_losses.append(s / len(train_ds))
    # Val
    model.eval()
    s = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            s += crit(model(xb).squeeze(), yb).item() * len(xb)
    val_losses.append(s / len(val_ds))

    if val_losses[-1] < best_val:
        best_val = val_losses[-1]
        torch.save(model.state_dict(), ARTIFACTS / 'best_gru.pt')
        best_ep = ep

    if ep % 5 == 0 or ep == 1:
        print(f"Epoch {ep:3d}/{EPOCHS}  train_mse={train_losses[-1]:.6f}  "
              f"val_mse={val_losses[-1]:.6f}")

print(f"\nBest val MSE = {best_val:.6f} at epoch {best_ep}")

Epoch   1/30  train_mse=0.870642  val_mse=1.163882
Epoch   5/30  train_mse=0.177933  val_mse=0.196271
Epoch  10/30  train_mse=0.158286  val_mse=0.220595
Epoch  15/30  train_mse=0.153639  val_mse=0.221977
Epoch  20/30  train_mse=0.150666  val_mse=0.249994
Epoch  25/30  train_mse=0.126510  val_mse=0.216540
Epoch  30/30  train_mse=0.109471  val_mse=0.168578

Best val MSE = 0.168578 at epoch 30


In [12]:
# Кривые обучения
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train MSE')
ax.plot(val_losses,   label='Val MSE')
ax.axvline(best_ep-1, ls='--', c='grey', label=f'Best epoch {best_ep}')
ax.legend(); ax.set(xlabel='Epoch', ylabel='MSE', title='GRU Learning Curves')
ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES / 'gru_learning_curves.png', dpi=150)
plt.close(fig)
print("Saved gru_learning_curves.png")

Saved gru_learning_curves.png


In [13]:
# Прогноз GRU на val
model.load_state_dict(torch.load(ARTIFACTS / 'best_gru.pt', map_location=device))
model.eval()

preds_sc = []
with torch.no_grad():
    for xb, _ in val_loader:
        xb = xb.to(device)
        preds_sc.append(model(xb).squeeze().cpu().numpy())
preds_sc = np.concatenate(preds_sc)

y_pred_gru = tgt_scaler.inverse_transform(preds_sc.reshape(-1,1)).flatten()
y_val_gru  = val_df['target'].values  

preds_test_sc = []
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device)
        preds_test_sc.append(model(xb).squeeze().cpu().numpy())
preds_test_sc = np.concatenate(preds_test_sc)
y_pred_gru_test = tgt_scaler.inverse_transform(preds_test_sc.reshape(-1,1)).flatten()
y_test_gru = test_df['target'].values

try:
    best_epoch = best_ep
except NameError:
    best_epoch = EPOCHS

log_exp('R1', 'GRU (window=72)', y_val_gru, y_pred_gru, y_test_gru, y_pred_gru_test,
        window_size=72, scaler='StandardScaler() on target', optimizer='Adam',
        lr=LR, epochs_trained=best_epoch, model_summary='GRU(64, layers=2) + Linear')


[R1] GRU (window=72)           | Val MAE=5.9324  RMSE=7.5882  MAPE=0.0393


## 6. Сравнение результатов

In [14]:
res_df = pd.DataFrame(experiments)
res_df.to_csv(ARTIFACTS / 'runs.csv', index=False)
print(res_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#4c72b0', '#55a868', '#c44e52', '#8172b2']
bars = ax.bar(res_df['experiment_id'], res_df['best_val_mae'], color=colors)
for bar, val in zip(bars, res_df['best_val_mae']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.2f}', ha='center', fontsize=10)
ax.set(title='Validation MAE: B1–B3 vs R1', ylabel='MAE')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES / 'baselines_compare.png', dpi=150)
plt.close(fig)
print("Saved baselines_compare.png")


experiment_id        task            dataset  seed                    split_summary window_size  horizon              model_summary                                                                          features_summary                     scaler optimizer     lr epochs_trained  best_val_mae  best_val_rmse  best_val_mape  test_mae  test_rmse  test_mape notes
           B1 forecasting S12-hw-dataset.csv    42 Train: 3024, Val: 648, Test: 648           1        1           Naive Last Value                                                                                     lag_1                                                                   6.4448         8.2010       0.043979    6.3424     8.0591   0.041485      
           B2 forecasting S12-hw-dataset.csv    42 Train: 3024, Val: 648, Test: 648          24        1             Moving Average                                                                           rolling_mean_24                                                     

## 7. Финальная оценка лучшей модели на Test

Test-выборка используется **ровно один раз**, для финальной оценки.

In [15]:
# Лучшая модель  best_val_mae
best_row = res_df.loc[res_df['best_val_mae'].idxmin()]
print(f"Best model: {best_row['experiment_id']} — {best_row['model_summary']}")

b3_row = res_df[res_df['experiment_id'] == 'B3'].iloc[0]
r1_row = res_df[res_df['experiment_id'] == 'R1'].iloc[0]

print(f"\nTest Results — Ridge (B3):  MAE={b3_row['test_mae']:.4f}  RMSE={b3_row['test_rmse']:.4f}  MAPE={b3_row['test_mape']:.4f}")
print(f"Test Results — GRU   (R1):  MAE={r1_row['test_mae']:.4f}  RMSE={r1_row['test_rmse']:.4f}  MAPE={r1_row['test_mape']:.4f}")


y_test_ridge = y_pred_test_b3

# Выбираем лучшую по test MAE для графика
if b3_row['test_mae'] <= r1_row['test_mae']:
    best_name, best_pred, best_true, best_dates = 'Ridge (B3)', y_test_ridge, y_test, test_f['date'].values
else:
    best_name, best_pred, best_true, best_dates = 'GRU (R1)', y_pred_gru_test, y_test_gru, test_df['date'].values


Best model: B3 — Ridge(alpha=1.0)

Test Results — Ridge (B3):  MAE=4.4057  RMSE=5.7000  MAPE=0.0285
Test Results — GRU   (R1):  MAE=7.1410  RMSE=8.9995  MAPE=0.0451


In [16]:
# лучший график 
fig, ax = plt.subplots(figsize=(15, 5))
n_show = min(300, len(best_true))
ax.plot(range(n_show), best_true[-n_show:],  label='True', lw=0.9)
ax.plot(range(n_show), best_pred[-n_show:],  label=f'{best_name} forecast', lw=0.9, ls='--')
ax.legend(); ax.set(title=f'Best Forecast on Test ({best_name})',
                    xlabel='Hours (last 300)', ylabel='Target')
ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES / 'best_forecast_test.png', dpi=150)
plt.close(fig)
print("Saved best_forecast_test.png")

Saved best_forecast_test.png


## 8. Обсуждение утечек данных Data Leakage

1. **Temporal split** — данные разбиты хронологически, будущее не попадает в train.
2. **Lag/rolling признаки** используют только прошлые значения (`shift ≥ 1`).
   `rolling_mean` считается **после shift(1)** — текущее наблюдение не включено.
3. **StandardScaler** обучен (`fit`) **только на train**; на val/test применяется `transform()`.
4. **Потенциальный риск:** если бы мы использовали `fit_transform` на всём датасете до
   разбиения, среднее и std содержали бы информацию о будущих значениях  leakage.
   В нашей реализации это **исключено**.

In [17]:
# сводка 
print(res_df.to_string(index=False))
print(f"\nBest on Test: {best_name}")
if b3_row['test_mae'] <= r1_row['test_mae']:
    print(f"  MAE={b3_row['test_mae']:.4f}  RMSE={b3_row['test_rmse']:.4f}  MAPE={b3_row['test_mape']:.4f}")
else:
    print(f"  MAE={r1_row['test_mae']:.4f}  RMSE={r1_row['test_rmse']:.4f}  MAPE={r1_row['test_mape']:.4f}")
print("\nArtifacts saved:")
for p in sorted(ARTIFACTS.rglob('*')):
    if p.is_file():
        print(f"  {p}")


experiment_id        task            dataset  seed                    split_summary window_size  horizon              model_summary                                                                          features_summary                     scaler optimizer     lr epochs_trained  best_val_mae  best_val_rmse  best_val_mape  test_mae  test_rmse  test_mape notes
           B1 forecasting S12-hw-dataset.csv    42 Train: 3024, Val: 648, Test: 648           1        1           Naive Last Value                                                                                     lag_1                                                                   6.4448         8.2010       0.043979    6.3424     8.0591   0.041485      
           B2 forecasting S12-hw-dataset.csv    42 Train: 3024, Val: 648, Test: 648          24        1             Moving Average                                                                           rolling_mean_24                                                     